# CHEM E 480 LAB 5

## NOTEBOOK OBJECTIVES

In this lab, you will:

* Implement a proportional controller
* Observe the behavior of a proportional controller
* Tune a proportional controller

## PROPORTIONAL CONTROLLER

There are different kinds of controllers that can be used to control the output of a system by taking note of the error between the set point and the measured value, and making changes in the input accordingly. For this lab, we will be looking at proportional control, where the magnitude of the adjustment made to the input is proportional to the error in the output (hence the name). 

Design and implement a proportional controller on the first heat sink. You should experiment with various control gain settings. Try controlling the temperature of the heat sink at various levels while you make set point changes. Introduce disturbances (e.g. using a fan to add convection, turning on the Q2 heater). Demonstrate overdamped and underdamped results.

Make sure that you don't send a signal higher than 100% to the heater!

You can mess around with different controller gain values, or you can use the ITAE tuning correlation and the results from your FOPTD model to tune the proportional controller:

$$K_c = \frac{0.20}{K_p}\bigg( \frac{\tau_p}{\theta_p} \bigg)^{1.22} $$

For each demonstration, plot (1) the setpoint, (2) the temperature, (3) the error, and (4) the heater output.

The output of a proportional controller is proportional to the error:

$$p(t) = \bar{p} + K_ce(t) $$

where $e(t)$ is the deviation from the setpoint and in this case $p$ is the heater level $Q$. You just need to make the signal you send to the heater `lab.Q1` proportional to the error.

### IMPORTS

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tclab
import time

### ADJUST CONTROLLER GAIN $K_c$ FROM ITAE TUNING CORRELATION

In [ ]:
# Calculate or pick a value for Kc
Kc = # FILL IN VALUE

# Set up time arrays in heatrs
n = # TIME POINTS  # Number of second time points (10 min)
tm = np.linspace(0,n-1,n) # Time values
lab = # INITIATE LAB
T1 = np.zeros(n)
Q1 = np.zeros(n)

# Step setpoint from 23.0 to 60.0 degC after 10 seconds
SP1 = np.ones(n)*#INITIAL SET POINT VALUE
SP1[# STEP AFTER 10 SECONDS] = # NEW SETPOINT VALUE

# Set Q1_bias (Q at steady state)
Q1_bias = # SET VALUE OF Q AT SS

# Run experiment
for i in range(n):
    
    # Record measurement
    T1[i] = # CALL T1 SENSOR
    
    # Fill-in P-only controller equation to change Q1[i]
    Q1[i] = # DEFINE WITH EQUATION ABOVE FOR CONTROLLING Q WITH P-CONTROL

    # Implement new heater value and clip to 0-100%
    Q1[i] = max(0,min(100,Q1[i]))
    
    # Set heater value
    lab.Q1(# SET HEATER VALUE TO CURRENT Q VALUE IN LOOP)
    
    # Every 20 values print column headers
    if i%20==0:
        
        print(' Heater,   Temp,  Setpoint')
    
    # Print values of Heater, Temp, and Setpoint
    print(f'{Q1[i]:7.2f},{T1[i]:7.2f},{SP1[i]:7.2f}')
    
    # wait for 1 sec
    time.sleep(1)
    
lab.close()

# Save data file
data = np.vstack((tm,Q1,T1,SP1)).T
np.savetxt('P-only.csv',data,delimiter=',',\
           header='Time,Q1,T1,SP1',comments='')

### PLOT DATA

Next we want to create a plot of our data, both the temperature data and value of heater power over time.

In [ ]:
# Create temperature figure
plt.figure(figsize=(10,7))
ax = plt.subplot(2,1,1)
ax.grid()
plt.plot(# TIME IN MINUTES,# SET POINT OF T1,'k-',label=r'$T_1$ SP')
plt.plot(# TIME IN MINUTES,# ACTUAL T1 READING,'r.',label=r'$T_1$ PV')
plt.ylabel(r'Temp ($^oC$)')
plt.legend(loc=2)

# Create heater value figure
ax = plt.subplot(2,1,2)
ax.grid()
plt.plot(# TIME IN MINUTES, # Q VALUE OVER TIME,'b-',label=r'$Q_1$')
plt.ylabel(r'Heater (%)')
plt.xlabel('Time (min)')
plt.legend(loc=1)
plt.savefig('P-only_Control.png')
plt.show()

## 2-PART SYSTEM

Add in a second proportional controller to maintain the temperatures of the second heat sink (T2) by manipulating the heater output of the second heater (Q2). Monitor the temperatures and the heater outputs of both systems. 

Does the steady-state Q1 value change for the same value of SP1 when the second heater is running?

In [ ]:
# Adjust controller gain (Kc) from ITAE tuning correlation
Kc = # SET VALUE

# Set up time arrays and heaters
n = # TIME POINTS  # Number of second time points (2.5 min)
tm = np.linspace(0,n-1,n) # Time values
lab = # CALL THE TEMPERATUER LAB
T1 = np.zeros(n)
T2 = np.zeros(n)
Q1 = np.zeros(n)
Q2 = np.zeros(n)

# Step setpoint from 23.0 to 70.0 degC
SP1 = np.ones(n)*23.0
SP1[# STEP UP AT 10 SEC AND ONWARD] = # NEW SET POINT VALUE

# Step setpoint from 23.0 to 55.0 degC
SP2 = np.ones(n)*23.0
SP2[# STEP UP AT 10 SEC AND ONWARD] = # NEW SET POINT VALUE

# Set Q1 and Q2 bias to 0
Q1_bias = # INITIAL Q1 VALUE
Q2_bias = # INITIAL Q2 VALUE

# Run experiment
for i in range(n):
    
    # Record temperature measurement for each
    T1[i] = # CALL TEMPERATURE SENSOR 1
    T2[i] = # CALL TEMPERATURE SENSOR 2
    
    # Fill-in P-only controller equation to change Q1[i] and Q2[i]
    Q1[i] = # SET EQUATION FOR P-CONTROL OF Q1 BASED ON T1 ERROR
    Q2[i] = # SET EQUATION FOR P-CONTROL OF Q2 BASED ON T2 ERROR
    
    # Implement new heater value for Q1 and clip to 0-100%
    Q1[i] = max(0,min(100,Q1[i]))
    lab.Q1(# SET NEW VALUE OF Q1)
    
    # Implement new heater value for Q2 and clip to 0-100%
    Q2[i] = max(0,min(100,Q2[i]))
    lab.Q2(# SET NEW VALUE OF Q2)
    
    # Print column headers every 20 data points
    if i%20==0:
        print(' Heater1,   Heater2   Temp1,     Temp2,  Setpoint')
   
    # Print heater 1, heater 2, temp 1, temp 2, setpoint
    print(f'{Q1[i]:7.2f},{Q2[i]:7.2f},{T1[i]:7.2f},{T2[i]:7.2f},{SP1[i]:7.2f},{SP2[i]:7.2f}')
    
    # Wait for 1 sec
    time.sleep(1)
    
lab.close()
# Save data file
# data = np.vstack((tm,Q1,T1,SP1)).T
# np.savetxt('P-only.csv',data,delimiter=',',\
#            header='Time,Q1,T1,SP1',comments='')

### PLOT DATA

Next we want to create a plot of our data, both the temperature data and value of heater power over time.

In [ ]:
# Create figure for temperature data
plt.figure(figsize=(10,7))
ax = plt.subplot(2,1,1)
ax.grid()

# Plot set point data
plt.plot(# TIME IN MINUTES,# SETPOINT OF T1 OVER TIME,'k-',label=r'$T_1$ SP')
plt.plot(# TIME IN MINUTES,# SET POINT OF T2 OVER TIME,'k-',label=r'$T_2$ SP')

# Plot temperature data
plt.plot(# TIME IN MINUTES,# T1 MEASURED OVER TIME,'r.',label=r'$T_1$ PV')
plt.plot(# TIME IN MINUTES,# T2 MEASURED OVER TIME,'g.',label=r'$T_2$ PV')
plt.ylabel(r'Temp ($^oC$)')
plt.legend(loc=2)

# Create figure for heater data
ax = plt.subplot(2,1,2)
ax.grid()
plt.plot(# TIME IN MINUTES,# HEATER 1 VALUE OVER TIME,'b-',label=r'$Q_1$')
plt.plot(# TIME IN MINUTES,# HEATER 2 VALUE OVER TIME,'g-',label=r'$Q_2$')
plt.ylabel(r'Heater (%)')
plt.xlabel('Time (min)')
plt.legend(loc=1)
plt.savefig('P-only_Control.png')
plt.show()